In [ ]:
import requests

def call_prediction_api(window_vals):
    url = "http://127.0.0.1:8002/predict"
    data = {"samples": window_vals.to_dict(orient='records')}
    response = requests.post(url, json=data)
    if response.status_code == 200:
        print("API Response:", response.json())
    else:
        print("API Error:", response.status_code, response.text)

In [ ]:

import pandas as pd
import time

data_path = r"/home/pitoleo/src/neat-calculator/neat_dashboard/merged_labeled_data_noisy.csv"

df = pd.read_csv(data_path)

# Keep the label column to save with windows
# df = df.drop(columns=['label'])


window = 128
step = 64
index = 0

X_windows = []


for start in range(0, len(df) - window + 1, step):
    end = start + window
    window_vals = df[start:end].copy()
    
    # Replace timestamps with current time + 20ms increments
    current_time_ms = int(time.time() * 1000)
    for i in range(len(window_vals)):
        window_vals.iloc[i, window_vals.columns.get_loc('timestamp')] = current_time_ms + (i * 20)
    
    # Add window_id to track which window this belongs to
    # window_vals['window_id'] = index
    
    X_windows.append(window_vals)
    
    call_prediction_api(window_vals)

    time.sleep(2.5)
    
    index += 1
    

# print(f"Created {len(X_windows)} windows of size {window} with step {step} out of {len(df)} data points.")

# # Concatenate all windows into a single DataFrame
# all_windows_df = pd.concat(X_windows, ignore_index=True)

# # Group by window_id and get the most prevalent label for each window
# if 'label' in all_windows_df.columns:
#     window_labels = all_windows_df.groupby('window_id')['label'].agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0])
#     window_summary = pd.DataFrame({
#         'window_id': window_labels.index,
#         'label': window_labels.values
#     })
# else:
#     window_summary = pd.DataFrame({
#         'window_id': all_windows_df['window_id'].unique()
#     })

# window_summary.to_csv(r"E:\src\neat-calculator\neat_dashboard\windows.csv", index=False)

# print(f"Saved {len(window_summary)} windows with their labels to windows.csv")


API Response: {'activity': 'WALKING_DOWNSTAIRS', 'confidence': 0.512884259223938, 'prediction_index': 4, 'all_probabilities': {'LAYING': 0.014949940145015717, 'SITTING': 0.023962223902344704, 'STANDING': 0.06292232125997543, 'WALKING': 0.09011100232601166, 'WALKING_DOWNSTAIRS': 0.512884259223938, 'WALKING_UPSTAIRS': 0.29517024755477905}}
API Response: {'activity': 'WALKING_DOWNSTAIRS', 'confidence': 0.49734148383140564, 'prediction_index': 4, 'all_probabilities': {'LAYING': 0.004808463156223297, 'SITTING': 0.007111404091119766, 'STANDING': 0.004759139381349087, 'WALKING': 0.3613750636577606, 'WALKING_DOWNSTAIRS': 0.49734148383140564, 'WALKING_UPSTAIRS': 0.12460443377494812}}
API Response: {'activity': 'WALKING_DOWNSTAIRS', 'confidence': 0.7419810891151428, 'prediction_index': 4, 'all_probabilities': {'LAYING': 0.002441615564748645, 'SITTING': 0.0031746469903737307, 'STANDING': 0.005602219607681036, 'WALKING': 0.02755521796643734, 'WALKING_DOWNSTAIRS': 0.7419810891151428, 'WALKING_UPSTA

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))